# ettm: Diffusion-TS unconditional

Sequence length is fixed to 128. Run cells from top to bottom. Training weights are shared across tasks.


In [ ]:
from pathlib import Path
import html, json, re, subprocess, sys, threading, time
from IPython.display import HTML, display
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while ROOT.name != "Diff-ts" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert ROOT.name == "Diff-ts", "Open this notebook from inside the Diff-ts repository."
NAME = "ettm"
TRUTH_NAME = "ettm"
CONFIG = ROOT / "Config/ettm/ettm_128.yaml"
SEQ_LEN = 128
GPU = 0
MILESTONE = 10
TASK = "uncond"
TRAIN_PROPORTION = 1.0
SEEDS = [0, 42, 123]
GPU_BY_SEED = {0: 1, 42: 2, 123: 3}
RUN_TRAINING = True  # Uncond starts training when the training cell runs.
RUN_SAMPLING = False  # Set True after the requested checkpoint exists.
print("repository:", ROOT)
print("config:", CONFIG)

def run_parallel_with_live_output(commands_by_seed, phase):
    """Run seed processes while replacing one bounded Jupyter status display."""
    startup_line_limit = 4
    display_line_limit = 500
    refresh_seconds = 0.5
    ansi_escape = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
    processes = {}
    threads = []
    startup_lines = {seed: [] for seed in commands_by_seed}
    latest_lines = {seed: "waiting for output..." for seed in commands_by_seed}
    states = {seed: "STARTING" for seed in commands_by_seed}
    status_lock = threading.Lock()
    status_changed = threading.Event()

    def clean(line):
        line = ansi_escape.sub("", line).replace("\b", "").strip()
        if len(line) > display_line_limit:
            line = "..." + line[-display_line_limit:]
        return line

    def status_display():
        with status_lock:
            lines = [f"{phase} live status (updated in place; full output is not saved)"]
            for seed in commands_by_seed:
                gpu = GPU_BY_SEED[seed]
                lines.append(f"\n[{phase} | seed {seed} | GPU {gpu}] {states[seed]}")
                if startup_lines[seed]:
                    lines.append("  startup:")
                    lines.extend(f"    {line}" for line in startup_lines[seed])
                lines.append(f"  current: {latest_lines[seed]}")
        body = html.escape("\n".join(lines))
        return HTML(f'<pre style="white-space:pre-wrap; margin:0">{body}</pre>')

    def record(seed, line):
        with status_lock:
            if len(startup_lines[seed]) < startup_line_limit:
                startup_lines[seed].append(line)
            latest_lines[seed] = line
        status_changed.set()

    def stream(seed, gpu, process):
        # Treat both newline and carriage return as record boundaries so tqdm-style
        # progress is visible without forwarding every update to the notebook.
        buffer = []
        while True:
            char = process.stdout.read(1)
            if char == "":
                if buffer:
                    line = clean("".join(buffer))
                    if line:
                        record(seed, line)
                break
            if char in "\r\n":
                if buffer:
                    line = clean("".join(buffer))
                    buffer.clear()
                    if line:
                        record(seed, line)
            else:
                buffer.append(char)
                if len(buffer) > display_line_limit * 2:
                    del buffer[:display_line_limit]
        process.stdout.close()

    for seed, command in commands_by_seed.items():
        gpu = GPU_BY_SEED[seed]
        process = subprocess.Popen(
            command,
            cwd=ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        processes[seed] = process
        states[seed] = "RUNNING"
        thread = threading.Thread(target=stream, args=(seed, gpu, process), daemon=True)
        thread.start(); threads.append(thread)

    handle = display(status_display(), display_id=True)
    return_codes = {}
    last_refresh = 0.0
    while len(return_codes) < len(processes) or any(thread.is_alive() for thread in threads):
        changed = status_changed.wait(timeout=0.1)
        status_changed.clear()

        for seed, process in processes.items():
            code = process.poll()
            if code is not None and seed not in return_codes:
                return_codes[seed] = code
                with status_lock:
                    states[seed] = "DONE" if code == 0 else f"FAILED (exit={code})"
                changed = True

        now = time.monotonic()
        if changed and now - last_refresh >= refresh_seconds:
            handle.update(status_display())
            last_refresh = now

    for thread in threads: thread.join()
    handle.update(status_display())
    if any(code != 0 for code in return_codes.values()):
        raise RuntimeError(f"{phase} failed: {return_codes}")
    return return_codes


## 1. Train the shared unconditional diffusion model

This is the README `Training` command. Set `RUN_TRAINING=True` above to execute it.


In [ ]:
def training_command(seed=None):
    gpu = GPU_BY_SEED[seed] if seed is not None else GPU
    cmd = [sys.executable, "-u", "main.py", "--name", NAME, "--config_file", str(CONFIG), "--gpu", str(gpu), "--task", TASK, "--train"]
    if seed is not None:
        cmd += ["--seed", str(seed), "--run_id", f"seed_{seed}"]
    cmd += ["dataloader.train_dataset.params.proportion", str(TRAIN_PROPORTION)]
    return cmd

training_seeds = SEEDS if TASK == "uncond" else [None]
train_commands = [training_command(seed) for seed in training_seeds]
for train_cmd in train_commands: print(" ".join(train_cmd))
if RUN_TRAINING:
    if TASK == "uncond":
        run_parallel_with_live_output(dict(zip(SEEDS, train_commands)), "TRAIN")
        for seed in SEEDS:
            checkpoint_dir = ROOT / "artifacts" / "uncond" / NAME / f"seed_{seed}" / "checkpoints_128"
            final_checkpoint = checkpoint_dir / f"checkpoint-{MILESTONE}.pt"
            assert final_checkpoint.exists(), f"Final checkpoint missing: {final_checkpoint}"
            for milestone in range(1, MILESTONE):
                old_checkpoint = checkpoint_dir / f"checkpoint-{milestone}.pt"
                if old_checkpoint.exists(): old_checkpoint.unlink()
            print(f"seed {seed}: retained checkpoint-{MILESTONE}.pt and removed checkpoints 1-{MILESTONE - 1}")
    else:
        subprocess.run(train_commands[0], cwd=ROOT, check=True)


## 2. Unconstrained sampling

This is the README `Unconstrained` command.


In [ ]:
def sampling_command(seed):
    return [sys.executable, "-u", "main.py", "--name", NAME, "--config_file", str(CONFIG), "--gpu", str(GPU_BY_SEED[seed]),
            "--seed", str(seed), "--run_id", f"seed_{seed}", "--sample", "0", "--milestone", str(MILESTONE)]

sample_commands = [sampling_command(seed) for seed in SEEDS]
for sample_cmd in sample_commands: print(" ".join(sample_cmd))
if RUN_SAMPLING:
    run_parallel_with_live_output(dict(zip(SEEDS, sample_commands)), "SAMPLE")


## 3. Evaluate and visualize generated series


In [ ]:
runs, metrics = {}, []
truth_file = f"sine_ground_truth_{SEQ_LEN}_train.npy" if TRUTH_NAME == "sine" else f"{TRUTH_NAME}_norm_truth_{SEQ_LEN}_train.npy"
for seed in SEEDS:
    artifact = ROOT / "artifacts" / "uncond" / NAME / f"seed_{seed}"
    generated_path = artifact / f"ddpm_fake_{NAME}.npy"
    truth_path = artifact / "samples" / truth_file
    assert generated_path.exists(), f"Run seed {seed} sampling first: {generated_path}"
    assert truth_path.exists(), f"Missing training truth: {truth_path}"
    generated, truth = np.load(generated_path), np.load(truth_path)
    n = min(len(generated), len(truth)); generated, truth = generated[:n], truth[:n]
    runs[seed] = (generated, truth)
    metrics.append({"seed": seed, "mean_error": float(abs(generated.mean()-truth.mean())),
                    "std_error": float(abs(generated.std()-truth.std()))})
for row in metrics: print(row)
print("mean across seeds:", {key: float(np.mean([row[key] for row in metrics])) for key in ("mean_error", "std_error")})

fig, axes = plt.subplots(len(SEEDS), 2, figsize=(14, 3.5 * len(SEEDS)))
for row, seed in enumerate(SEEDS):
    generated, truth = runs[seed]
    axes[row, 0].plot(truth[0, :, 0], label="real"); axes[row, 0].plot(generated[0, :, 0], label="generated", alpha=.8)
    axes[row, 0].set_title(f"seed {seed}: feature 0 sample path"); axes[row, 0].legend()
    axes[row, 1].hist(truth[..., 0].ravel(), bins=60, density=True, alpha=.5, label="real")
    axes[row, 1].hist(generated[..., 0].ravel(), bins=60, density=True, alpha=.5, label="generated")
    axes[row, 1].set_title(f"seed {seed}: feature 0 distribution"); axes[row, 1].legend()
plt.tight_layout()


## 4. Generation quality metrics

Compute context-FID, correlational, discriminative, and predictive scores for every seed. Lower is better for all four metrics.


In [ ]:
from evaluation.generative_metrics import evaluate_generation

metric_results = {}
for seed in SEEDS:
    generated, truth = runs[seed]
    print(f"Evaluating seed {seed} on GPU {GPU_BY_SEED[seed]} ...")
    scores = evaluate_generation(truth, generated, gpu=GPU_BY_SEED[seed], seed=seed)
    metric_results[seed] = scores
    metric_path = ROOT / "artifacts" / "uncond" / NAME / f"seed_{seed}" / "metrics.json"
    metric_path.write_text(json.dumps(scores, indent=2) + "\n")
    print(seed, scores)

metric_names = ("context_fid", "correlational_score", "discriminative_score", "predictive_score")
metric_summary = {
    metric: {
        "mean": float(np.mean([metric_results[seed][metric] for seed in SEEDS])),
        "std": float(np.std([metric_results[seed][metric] for seed in SEEDS], ddof=1)),
    }
    for metric in metric_names
}
summary_path = ROOT / "artifacts" / "uncond" / NAME / "metrics_summary.json"
summary_path.write_text(json.dumps({"seeds": metric_results, "summary": metric_summary}, indent=2) + "\n")
print(json.dumps(metric_summary, indent=2))
print("saved:", summary_path)
